In [11]:
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(),'..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
from src.evaluator.word_vector import concept_arithmetic, corporate_arithmetic, quick_search
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


In [12]:
df = pd.read_csv('../data/raw/edinet_analysis_data_listed_only.csv')
print(df.shape)
df.head()


(3637, 9)


,company_name,証券コード,33業種コード,33業種区分,17業種コード,17業種区分,business_description,business_risks,filename
0,AGC株式会社,5201,3400,ガラス・土石製品,3,建設・資材,3 事業の内容 当社及び当社の子会社(以下、「当社グループ」という。)並びに当社の関連会社は...,3 事業等のリスク (1)リスクマネジメント体制 当社グループでは、「第4 提出会社の状況 ...,S100XSTU.zip
1,AGS株式会社,3648,5250,情報・通信業,10,情報通信・サービスその他,3 事業の内容 当社グループ(当社及び当社の関係会社)は、当社と連結子会社3社とで構成されて...,3 事業等のリスク 文中の将来に関する事項は、当連結会計年度末現在において当社グループが判断...,S100W007.zip
2,AHCグループ株式会社,7083,9050,サービス業,10,情報通信・サービスその他,3 事業の内容 当社グループは、当社、連結子会社(SLカンパニー株式会社、テラスワールド株式...,3 事業等のリスク 有価証券報告書に記載した事業の状況、経理の状況等に関する事項のうち、投資...,S100XMZ1.zip
3,AI CROSS株式会社,4476,5250,情報・通信業,10,情報通信・サービスその他,"3 事業の内容 当社グループは、「Smart Work, Smart Life~人生のいい時...",3 事業等のリスク 本書に記載した事業の状況、経理の状況等に関する事項のうち、投資家の判断に...,S100XUJV.zip
4,AI inside 株式会社,4488,5250,情報・通信業,10,情報通信・サービスその他,3 事業の内容 当社は、「AIで、人類の進化と人々の幸福に貢献する」というパーパスのもと、「...,3 事業等のリスク 当社は、事業展開上のリスクになる可能性があると考えられる主な要因として、...,S100W5FA.zip


In [ ]:
# テスト読み込み(ファインチューニング後の改良モデル)
test_embeddings = np.load('../models/true_neo_company_embeddings.npy')
test_df = pd.read_pickle('../models/true_neo_company_data_with_sbert.pkl')

print(f"ベクトルの形状: {test_embeddings.shape}") # (3637, 768) と出れば成功
print(f"データフレームの行数: {len(test_df)}")   # 3647 と出れば成功

ベクトルの形状: (3637, 768)
データフレームの行数: 3637


In [14]:
#関連する会社を検索できる
model = SentenceTransformer('intfloat/multilingual-e5-base')
res = quick_search("日焼けしたキティやシナモン、クロミたちがかわいい♪ 夏仕様のフィギュア付き食玩が登場", model, test_embeddings, test_df)

# company_name はインデックスにいるので、そのまま index と score を表示します
print(res['score'])

Loading weights: 100%|██████████████████████| 199/199 [00:00<00:00, 1029.25it/s]


company_name
株式会社アクアライン                                0.500295
CRAVIA株式会社(旧会社名 アジャイルメディア・ネットワーク株式会社)     0.499303
株式会社ダイキアクシス                               0.495692
株式会社Waqoo                                 0.490055
株式会社トランスジェニックグループ (旧会社名 株式会社トランスジェニック)    0.485547
ケイティケイ株式会社                                0.484915
ヒビノ株式会社                                   0.484099
株式会社キャンディル                                0.482546
株式会社ミツウロコグループホールディングス                     0.477620
株式会社NEXYZ.Group                           0.476041
Name: score, dtype: float32


In [15]:
# 実行(概念を加算・引算できる)
res = concept_arithmetic("KDDI株式会社", "キャラクター", "不動産", test_embeddings, test_df, model)

if res is not None:
    # インデックス（企業名）とスコアを表示
    print(res[['calc_score']])

                      calc_score
company_name                    
沖縄セルラー電話株式会社            0.973264
ソフトバンク株式会社              0.956857
株式会社スカパーJSATホールディングス    0.807123
株式会社ビジョン                0.804299
株式会社ワイヤレスゲート            0.798018
株式会社朝日ネット               0.785940
株式会社ジェノバ                0.785162
株式会社WOWOW               0.777637
ビーウィズ株式会社               0.768127
株式会社プロディライト             0.762799
